# S13 · A neural network, built by hand

A neural network is a flexible function with a lot of little dials on it. Feed
something in, a guess comes out, and the network learns by nudging its dials to
make the guesses less wrong.

Before we let PyTorch do the heavy lifting in the next notebook, we build the
pieces here by hand with NumPy: one neuron, the "bend" that lets a network fit
curves, and a full pass from input to prediction. Nothing here is magic once you
see the parts.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on
  each cell, top to bottom, and read the plain-English note above each one.
- New to the maths of "which way is downhill"? Open the primer
  `primers/slopes_and_gradients.md` for a ten-minute, picture-first version.
- Already confident with code or calculus? Skip to the cell marked
  **Stretch (optional)** near the end, where backprop and the chain rule live.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses only numpy and matplotlib, which Colab already ships.
# So there is nothing to install here.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                 # fast maths on lists of numbers
import matplotlib.pyplot as plt     # drawing charts

# Set a seed so everyone gets the same random numbers and the same pictures.
np.random.seed(0)

## Step 1 — what one neuron does

The smallest piece of a network is a **neuron**, and it does three tiny things:

1. multiply each input by a **weight** (its own dial for that input),
2. add the results up and add one more number called the **bias** (a baseline),
3. pass that total through a **bend** (we get to bends in Step 2).

Steps 1 and 2 together are just "multiply each input by its weight and add them
up". Let us do it once, by hand, for a neuron with three inputs.

In [ ]:
# Three inputs for one example (think: three measurements about something).
inputs = np.array([0.5, -1.0, 2.0])

# One weight per input (three dials), plus a single bias number.
weights = np.array([0.4, 0.2, -0.1])
bias = 0.05

# Multiply each input by its weight, add them up, then add the bias.
weighted_sum = np.dot(weights, inputs) + bias

print("inputs      :", inputs)
print("weights     :", weights)
print("bias        :", bias)
print("weighted sum:", round(weighted_sum, 4))

## Step 2 — the "bend" (activation)

The weighted sum is just a number. On its own, stacking these would only ever give
you a straight line. The trick that makes networks powerful is to pass the number
through a **bend**, called an **activation function**. Two famous ones:

- **ReLU**: `max(0, z)` — keep the number if it is positive, otherwise return 0.
- **sigmoid**: squashes any number into the range 0 to 1.

We write them as plain functions and draw them so you can see the shapes.

In [ ]:
# ReLU: keep the number if it is positive, otherwise 0.
def relu(z):
    return np.maximum(0, z)


# Sigmoid: squash any number into the range between 0 and 1.
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


# A range of input values to draw the curves.
z_values = np.linspace(-6, 6, 200)

plt.figure(figsize=(7, 5))
plt.plot(z_values, relu(z_values), color="#2E75B6", linewidth=3, label="ReLU")
plt.plot(z_values, sigmoid(z_values), color="#C0392B", linewidth=3, label="sigmoid")
plt.axhline(0, color="grey", linewidth=0.8)
plt.axvline(0, color="grey", linewidth=0.8)
plt.xlabel("z  (the weighted sum from Step 1)")
plt.ylabel("output after the bend")
plt.title("Two common activation functions")
plt.legend()
plt.show()

## Step 3 — finish the single neuron

Now pass our weighted sum from Step 1 through a bend. That final number is the
neuron's output. That is a complete neuron: multiply, add, then bend.

In [ ]:
# Pass the weighted sum through ReLU to get the neuron's output.
neuron_output = relu(weighted_sum)

print("weighted sum:", round(weighted_sum, 4))
print("after ReLU  :", round(neuron_output, 4))
print()
print("That is one neuron: multiply, add, then bend.")

## Step 4 — why the bend matters

Here is the key idea of the whole session. If you stack layers with **no** bend
between them, the result is *still just a straight line*. You gained nothing by
going deeper. The bend is what lets the network curve and kink to fit real,
messy data. Let us see it with a tiny example: on the left, two straight steps
with no bend; on the right, the same idea but with a ReLU in the middle.

In [ ]:
# Make some input values.
x_line = np.linspace(-3, 3, 100)

# Two straight steps with NO bend in between: a line of a line is still a line.
first_step = 1.6 * x_line + 0.5
stacked_straight = 0.7 * first_step - 0.4

# The same idea but with ReLU bends: now we can make a curve with kinks.
bent = relu(x_line + 1.5) - 2 * relu(x_line) + relu(x_line - 1.5)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x_line, stacked_straight, color="#C0392B", linewidth=3)
axes[0].set_title("No bend:\nstill just a straight line")
axes[1].plot(x_line, bent, color="#2E75B6", linewidth=3)
axes[1].set_title("With ReLU:\nwe can bend and fit curves")
plt.show()

print("Lesson: the bend is what makes depth worth having.")

## Step 5 — a whole layer at once

A **layer** is just many neurons side by side, each with its own weights, all
looking at the same inputs. Instead of one weight list we now have a table of
weights (one row per neuron), and the weighted sums become a table-times-list
step. If tables of numbers are new, `primers/vectors_and_matrices.md` has the
picture.

Our tiny network: 3 inputs → a hidden layer of 4 neurons (each with a ReLU) → 1
output number. Sending data through, left to right, is called a **forward pass**.

In [ ]:
# One example with 3 input features.
one_example = np.array([0.5, -1.0, 2.0])

# Hidden layer: 4 neurons, each with its own weights for the 3 inputs.
# So the weight table has shape (4, 3): one row of weights per neuron.
hidden_weights = np.random.normal(0, 0.5, size=(4, 3))
hidden_bias = np.zeros(4)

# The weighted sums for all 4 hidden neurons at once ( @ means table times list).
hidden_sum = hidden_weights @ one_example + hidden_bias

# Bend each hidden neuron with ReLU.
hidden_output = relu(hidden_sum)

print("hidden weighted sums:", hidden_sum.round(3))
print("hidden after ReLU   :", hidden_output.round(3))

## Step 6 — the output layer

The output layer takes the 4 hidden numbers and combines them into one final
prediction, with one more weighted sum. For a single plain number out, we do not
add a bend here.

In [ ]:
# Output layer: 1 neuron looking at the 4 hidden outputs.
# So its weight table has shape (1, 4).
output_weights = np.random.normal(0, 0.5, size=(1, 4))
output_bias = np.zeros(1)

prediction = output_weights @ hidden_output + output_bias

print("final prediction:", prediction.round(4))
print()
print("That is a full forward pass: input -> hidden (ReLU) -> output.")

### Stretch (optional) — peek inside "learning": backprop is the chain rule

Skip this if you are new to calculus; the next notebook trains a real network
without any of it. But if you want to see what "the network learns" actually does,
here it is, stripped to the bone.

To improve a dial, we need to know how the mistake changes when we nudge that
dial. That "how the mistake changes per nudge" number is the **gradient**.
**Backpropagation** is just the **chain rule** from calculus, worked step by step.

Take a tiny one-dial chain:

- `z = w * x`      (a linear step)
- `h = z ** 2`     (a nonlinear step, standing in for a bend)
- `L = (h - y)**2` (the squared mistake, our loss)

The chain rule says: multiply the little slopes along the path,
`dL/dw = dL/dh * dh/dz * dz/dw`. Let us compute each piece and multiply.

In [ ]:
# Fixed numbers for this tiny example.
x = 2.0    # the input
w = 0.5    # the one dial we want a gradient for
y = 3.0    # the true target

# Forward pass: compute the value step by step.
z = w * x
h = z ** 2
L = (h - y) ** 2

print("forward pass:")
print("  z =", z)
print("  h =", h)
print("  L =", round(L, 4))

# Backward pass: the little slope of each step.
dL_dh = 2 * (h - y)   # slope of (h - y)^2 with respect to h
dh_dz = 2 * z         # slope of z^2 with respect to z
dz_dw = x             # slope of w*x with respect to w

# Chain rule: multiply the little slopes along the path.
dL_dw = dL_dh * dh_dz * dz_dw

print()
print("backward pass (chain rule):")
print("  dL/dh =", dL_dh)
print("  dh/dz =", dh_dz)
print("  dz/dw =", dz_dw)
print("  dL/dw = dL/dh * dh/dz * dz/dw =", round(dL_dw, 4))

### Stretch (optional) — check that gradient by nudging

We can sanity-check the gradient with the plain meaning of a slope: nudge `w` by a
tiny amount and see how much the loss `L` changes. The two numbers should match
closely. This is exactly the bookkeeping PyTorch's autograd will do for us
automatically from the next notebook on, so we never hand-derive a gradient
again.

In [ ]:
# A helper that runs the whole forward pass for a given weight.
def loss_for_weight(weight_value):
    z = weight_value * x
    h = z ** 2
    return (h - y) ** 2


# Nudge w by a tiny step and measure how the loss changes.
tiny_step = 1e-6
loss_now = loss_for_weight(w)
loss_nudged = loss_for_weight(w + tiny_step)
numerical_gradient = (loss_nudged - loss_now) / tiny_step

print("chain-rule gradient:", round(dL_dw, 4))
print("nudge gradient     :", round(numerical_gradient, 4))
print()
print("They match - backprop really is just the chain rule.")

## What you just did

You built every piece of a neural network with plain NumPy: a neuron (multiply,
add, bend), the ReLU and sigmoid bends, a hidden layer written as a table-times-
list, and a full forward pass from input to prediction. In the Stretch cells you
also watched "learning" up close: one chain-rule gradient, checked by nudging.

Next notebook: `02_train_a_neural_network.ipynb`, where PyTorch does all of this
for us at scale and we train a real network that reads handwritten digits.